# Can we predict a warranty claim within a year?
Follows protocol.yaml: fixed holdout (holdout.csv), 5-fold stratified CV on training rows, selection by mean CV roc_auc, holdout scored once.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf'
df=pd.read_csv('data.csv'); print(df.shape); df.describe(include='all')

In [ ]:
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(hold_idx)==1400 and hold_idx.max()<len(df)
mask=np.zeros(len(df),bool); mask[hold_idx]=True
target='claim_within_year'
# device_serial excluded (identifier); all other columns are known at sale/registration
cat=['channel']; num=['device_age_years','price_eur','daily_hours']; features=cat+num
train=df[~mask]; hold=df[mask]
X,y=train[features],train[target]; Xh,yh=hold[features],hold[target]
print(len(train),len(hold),y.mean(),yh.mean())
print(train.isna().sum())

In [ ]:
def prep(scale):
    n=[SimpleImputer(strategy='median')]+([StandardScaler()] if scale else [])
    return ColumnTransformer([('c',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(handle_unknown='ignore')),cat),
                              ('n',make_pipeline(*n),num)])
# all candidates named before any is scored
candidates={
 'baseline_prior':make_pipeline(prep(False),DummyClassifier(strategy='prior')),
 'logreg_C1':make_pipeline(prep(True),LogisticRegression(C=1.0,max_iter=1000)),
 'logreg_C0.1':make_pipeline(prep(True),LogisticRegression(C=0.1,max_iter=1000)),
 'random_forest':make_pipeline(prep(False),RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1)),
 'hist_gb':make_pipeline(prep(False),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)),
}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=0)
cv_scores={k:cross_val_score(m,X,y,cv=cv,scoring='roc_auc').tolist() for k,m in candidates.items()}  # preprocessing is inside the pipeline, so fitted per fold
summary=pd.Series({k:np.mean(v) for k,v in cv_scores.items()}).sort_values(ascending=False); summary

In [ ]:
chosen=summary.index[0]; print('chosen:',chosen)
model=candidates[chosen].fit(X,y)
holdout_auc=roc_auc_score(yh,model.predict_proba(Xh)[:,1])  # scored once
print('holdout roc_auc',holdout_auc)
cv_folds=[[[int(train.index[i]) for i in tr],[int(train.index[i]) for i in te]] for tr,te in cv.split(X,y)]  # [train, test], 0-based positions in data.csv
json.dump({'features':features,'candidates':list(candidates),'cv_folds':cv_folds,'cv_scores':cv_scores,
 'chosen':chosen,'holdout_rows':[int(i) for i in hold.index],
 'holdout_scores':{chosen:float(holdout_auc)}},open('evidence.json','w'))